In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from airflow.providers.postgres.hooks.postgres import PostgresHook
from sqlalchemy import MetaData, Table, Column, Integer, inspect, UniqueConstraint, Float, Boolean

In [3]:
def extract():
    import os
    from dotenv import load_dotenv
    from sqlalchemy import create_engine
    load_dotenv()
    dst_host = os.environ.get('DB_DESTINATION_HOST')
    dst_port = os.environ.get('DB_DESTINATION_PORT')
    dst_username = os.environ.get('DB_DESTINATION_USER')
    dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
    dst_db = os.environ.get('DB_DESTINATION_NAME')
    conn=create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')
        #hook = PostgresHook('source_db')
        #conn = hook.get_conn()
    sql = f"""
        SELECT
                f.id,
                f.floor,
                f.kitchen_area,
                f.living_area,
                f.rooms,
                f.is_apartment,
                f.studio,
                f.total_area,
                f.price,
                b.build_year,
                b.building_type_int,
                b.latitude,
                b.longitude,
                b.ceiling_height,
                b.flats_count,
                b.floors_total,
                b.has_elevator
        FROM flats AS f
        LEFT JOIN buildings AS b ON b.id = f.building_id;
        """
    data = pd.read_sql(sql, conn)
    return data

In [4]:
def transform(data: pd.DataFrame):
        #Удаляем дубли
    feature_cols = [col for col in data.columns if col != 'id']
    is_duplicated_features = data.duplicated(subset=feature_cols, keep=False)
    data = data[~is_duplicated_features].reset_index(drop=True)
        
        #Удаляем выбросы
    num_cols = ['total_area','price','flats_count']
    threshold = 1.5
    potential_outliers = pd.DataFrame()

    for col in num_cols:
        Q1 = data[col].quantile(0.25)# Ваш код здесь #
        Q3 = data[col].quantile(0.75)# Ваш код здесь #
        IQR = Q3 - Q1# Ваш код здесь #
        margin = threshold*IQR # Ваш код здесь #
        lower = Q1 - margin # Ваш Код здесь #
        upper = Q3 + margin # Ваш Код здесь #
        potential_outliers[col] = ~data[col].between(lower, upper)

    outliers = potential_outliers.any(axis=1)
    data=data[~outliers].reset_index(drop=True)
        
    return data

In [5]:
transform(extract())

,id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,9,9.90,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,7,0.00,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,9,9.00,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,4,3,3.00,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True
4,5,9,0.00,0.000000,2,False,False,51.009998,8490104,2017,4,55.724728,37.743069,2.70,192,17,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105960,141356,8,6.00,42.000000,3,False,False,64.000000,10800000,1971,4,55.740402,37.834579,2.64,428,9,True
105961,141358,5,5.28,28.330000,2,False,False,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,False
105962,141359,7,5.30,20.000000,1,False,False,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,True
105963,141360,15,13.80,33.700001,2,False,False,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,True
